In [1]:
import pandas as pd
import numpy as np

# ---- PARAMETERS ----
np.random.seed(42)  # for reproducibility of synthetic data

regions = ["UK", "US"]

# Months to simulate
start_date = "2021-01-01"
end_date = "2023-12-31"
months = pd.date_range(start=start_date, end=end_date, freq="MS")

# Demographic categories
urban_rural = ["Urban", "Rural"]
income_levels = ["Low", "Middle", "High"]
age_groups = ["18-24", "25-34", "35-44", "45-54", "55+"]
genders = ["Male", "Female", "Other"]

# ---- SYNTHETIC DATA GENERATION ----
rows = []
for region in regions:
    for month_start in months:
        year = month_start.year
        month = month_start.month
        # synthetic monthly sales revenue in GBP for UK, USD for US – placeholder
        base_sales = 100_000 if region == "UK" else 120_000
        # vary by season and random noise
        season_factor = 1 + 0.1 * np.sin((month - 1) / 12 * 2 * np.pi)
        monthly_sales = base_sales * season_factor * np.random.uniform(0.8, 1.2)
        # profit margin as percentage
        profit_margin = np.random.uniform(0.08, 0.18)  # 8% to 18% typical synthetic range
        
        # demographic shares (must sum to 1 across each category)
        # Example: urban/rural split
        urban_share = np.random.beta(5, 2)  # bias toward urban
        rural_share = 1 - urban_share
        urban_rural_shares = {"Urban": urban_share, "Rural": rural_share}
        
        # Income shares
        income_raw = np.random.dirichlet([2, 5, 2])  # low/mid/high
        income_shares = dict(zip(income_levels, income_raw))
        
        # Age shares
        age_raw = np.random.dirichlet([1, 2, 2, 1, 1])
        age_shares = dict(zip(age_groups, age_raw))
        
        # Gender shares
        gender_raw = np.random.dirichlet([4, 4, 0.5])
        gender_shares = dict(zip(genders, gender_raw))
        
        row = {
            "Region": region,
            "Year": year,
            "Month": month,
            "Date": month_start,
            "Monthly_Sales": round(monthly_sales, 2),
            "Monthly_Profit_Margin": round(profit_margin, 4),
            # Demographic shares
            "Urban_Share": round(urban_rural_shares["Urban"], 4),
            "Rural_Share": round(urban_rural_shares["Rural"], 4),
            "Income_Low_Share": round(income_shares["Low"], 4),
            "Income_Middle_Share": round(income_shares["Middle"], 4),
            "Income_High_Share": round(income_shares["High"], 4),
            "Age_18_24_Share": round(age_shares["18-24"], 4),
            "Age_25_34_Share": round(age_shares["25-34"], 4),
            "Age_35_44_Share": round(age_shares["35-44"], 4),
            "Age_45_54_Share": round(age_shares["45-54"], 4),
            "Age_55plus_Share": round(age_shares["55+"], 4),
            "Gender_Male_Share": round(gender_shares["Male"], 4),
            "Gender_Female_Share": round(gender_shares["Female"], 4),
            "Gender_Other_Share": round(gender_shares["Other"], 4),
        }
        rows.append(row)

# ---- BUILD DATAFRAME ----
df_monthly = pd.DataFrame(rows)

# ---- AGGREGATE QUARTERLY ----
df_monthly["Quarter"] = df_monthly["Date"].dt.to_period("Q")
agg_funcs = {
    "Monthly_Sales": "sum",
    # Weighted average for profit margin: weight by sales
    "Monthly_Profit_Margin": lambda x: np.average(x, weights=df_monthly.loc[x.index, "Monthly_Sales"]),
    # For demographic shares, compute weighted average by sales
    "Urban_Share": lambda x: np.average(x, weights=df_monthly.loc[x.index, "Monthly_Sales"]),
    "Rural_Share": lambda x: np.average(x, weights=df_monthly.loc[x.index, "Monthly_Sales"]),
    "Income_Low_Share": lambda x: np.average(x, weights=df_monthly.loc[x.index, "Monthly_Sales"]),
    "Income_Middle_Share": lambda x: np.average(x, weights=df_monthly.loc[x.index, "Monthly_Sales"]),
    "Income_High_Share": lambda x: np.average(x, weights=df_monthly.loc[x.index, "Monthly_Sales"]),
    "Age_18_24_Share": lambda x: np.average(x, weights=df_monthly.loc[x.index, "Monthly_Sales"]),
    "Age_25_34_Share": lambda x: np.average(x, weights=df_monthly.loc[x.index, "Monthly_Sales"]),
    "Age_35_44_Share": lambda x: np.average(x, weights=df_monthly.loc[x.index, "Monthly_Sales"]),
    "Age_45_54_Share": lambda x: np.average(x, weights=df_monthly.loc[x.index, "Monthly_Sales"]),
    "Age_55plus_Share": lambda x: np.average(x, weights=df_monthly.loc[x.index, "Monthly_Sales"]),
    "Gender_Male_Share": lambda x: np.average(x, weights=df_monthly.loc[x.index, "Monthly_Sales"]),
    "Gender_Female_Share": lambda x: np.average(x, weights=df_monthly.loc[x.index, "Monthly_Sales"]),
    "Gender_Other_Share": lambda x: np.average(x, weights=df_monthly.loc[x.index, "Monthly_Sales"]),
}
df_quarterly = (
    df_monthly.groupby(["Region", "Quarter"])
    .agg(agg_funcs)
    .reset_index()
)
# rename aggregated columns for clarity
df_quarterly = df_quarterly.rename(
    columns={
        "Monthly_Sales": "Quarterly_Sales",
        "Monthly_Profit_Margin": "Quarterly_Profit_Margin",
    }
)

# ---- AGGREGATE ANNUAL ----
df_monthly["Year"] = df_monthly["Date"].dt.year
df_annual = (
    df_monthly.groupby(["Region", "Year"])
    .agg(agg_funcs)
    .reset_index()
)
df_annual = df_annual.rename(
    columns={
        "Monthly_Sales": "Annual_Sales",
        "Monthly_Profit_Margin": "Annual_Profit_Margin",
    }
)

# ---- SAVE TO CSV ----
df_monthly.to_csv("vive_iron_brew_monthly.csv", index=False)
df_quarterly.to_csv("vive_iron_brew_quarterly.csv", index=False)
df_annual.to_csv("vive_iron_brew_annual.csv", index=False)

print("CSV files created:")
print(" - vive_iron_brew_monthly.csv")
print(" - vive_iron_brew_quarterly.csv")
print(" - vive_iron_brew_annual.csv")

CSV files created:
 - vive_iron_brew_monthly.csv
 - vive_iron_brew_quarterly.csv
 - vive_iron_brew_annual.csv
